# Model Sanity Check

Load all three models, run predictions on 5 personas, verify the numbers in the report.

**Note on reproducibility:** The naive baseline and classical model use GDELT news popularity scores, which are computed from a rolling 30-day window with exponential recency decay. Because the news cycle shifts daily, exact numeric values will differ between runs. The report's numbers (R08) are a point-in-time snapshot. What should be stable across runs: the ranking of models (classical > DL on scrutiny), the naive baseline returning only `general_spending`, and the near-zero Jaccard overlap between classical and DL.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))
os.chdir("..")

import pandas as pd
import numpy as np

from scripts.naive_baseline import GDELTPopularityScorer, evaluate_baseline
from scripts.classical import TFIDFRecommender, evaluate
from scripts.deep_learning import HybridNeuralRecommender

contracts = pd.read_csv("data/processed/unified_contracts.csv")
print(f"Loaded {len(contracts):,} contracts, {contracts['topic'].nunique()} topics")

In [ ]:
PERSONAS = [
    {"name": "Healthcare advocate",  "topics": ["healthcare", "research"]},
    {"name": "Defense watcher",      "topics": ["defense", "government_efficiency"]},
    {"name": "Foreign aid critic",   "topics": ["foreign_aid", "general_spending"]},
    {"name": "Education advocate",   "topics": ["education", "infrastructure"]},
    {"name": "Fiscal accountability","topics": ["government_efficiency", "finance"]},
]
TOP_N = 20

## 1. Naive Baseline (GDELT Popularity)

In [ ]:
baseline = GDELTPopularityScorer("data/raw/gdelt_articles.csv")
baseline.fit()

naive_top20 = baseline.recommend(contracts, top_n=TOP_N)
print(f"Naive top-{TOP_N}: {naive_top20['topic'].nunique()} unique topics")
print(f"  Topics: {naive_top20['topic'].value_counts().to_dict()}")
print(f"  Mean scrutiny: {naive_top20['doge_scrutiny_score'].mean():.3f}")
print(f"  Mean value: ${naive_top20['value'].mean():,.0f}")

print("\nPer-persona precision:")
for p in PERSONAS:
    m = evaluate_baseline(naive_top20, p["topics"])
    print(f"  {p['name']}: precision@{TOP_N}={m['precision_at_k']:.3f}, coverage={m['topic_coverage']:.3f}")

## 2. Classical (TF-IDF + Citizen Impact)

In [ ]:
classical = TFIDFRecommender()
classical.fit(contracts)

classical_results = {}
for p in PERSONAS:
    recs = classical.recommend(p["topics"], top_n=TOP_N, alpha=0.7)
    classical_results[p["name"]] = recs

# Aggregate metrics
scrutiny_vals = [r["doge_scrutiny_score"].mean() for r in classical_results.values()]
value_vals = [r["value"].mean() for r in classical_results.values()]
diversity_vals = [r["topic"].nunique() for r in classical_results.values()]

print(f"Classical model (averaged across {len(PERSONAS)} personas):")
print(f"  Mean DOGE scrutiny: {np.mean(scrutiny_vals):.3f}")
print(f"  Mean contract value: ${np.mean(value_vals):,.0f}")
print(f"  Mean topic diversity: {np.mean(diversity_vals):.1f}")

print("\nPer-persona breakdown:")
for p in PERSONAS:
    r = classical_results[p["name"]]
    print(f"  {p['name']}:")
    print(f"    scrutiny={r['doge_scrutiny_score'].mean():.3f}, value=${r['value'].mean():,.0f}, topics={r['topic'].nunique()}")

## 3. Deep Learning (Sentence Transformer + MLP Ranker)

In [ ]:
dl = HybridNeuralRecommender(device="cpu")
dl.load_artifacts(contracts=contracts)

dl_results = {}
for p in PERSONAS:
    recs = dl.recommend(p["topics"], top_n=TOP_N)
    dl_results[p["name"]] = recs

scrutiny_vals = [r["doge_scrutiny_score"].mean() for r in dl_results.values()]
value_vals = [r["value"].mean() for r in dl_results.values()]
diversity_vals = [r["topic"].nunique() for r in dl_results.values()]

print(f"DL model (averaged across {len(PERSONAS)} personas):")
print(f"  Mean DOGE scrutiny: {np.mean(scrutiny_vals):.3f}")
print(f"  Mean contract value: ${np.mean(value_vals):,.0f}")
print(f"  Mean topic diversity: {np.mean(diversity_vals):.1f}")

print("\nPer-persona breakdown:")
for p in PERSONAS:
    r = dl_results[p["name"]]
    print(f"  {p['name']}:")
    print(f"    scrutiny={r['doge_scrutiny_score'].mean():.3f}, value=${r['value'].mean():,.0f}, topics={r['topic'].nunique()}")

## 4. Cross-Model Comparison

In [ ]:
# Summary table matching R08
rows = []
for label, results in [("Naive (GDELT)", {p["name"]: naive_top20 for p in PERSONAS}),
                        ("Classical (TF-IDF)", classical_results),
                        ("Deep Learning (MLP)", dl_results)]:
    s = [r["doge_scrutiny_score"].mean() for r in results.values()]
    v = [r["value"].mean() for r in results.values()]
    d = [r["topic"].nunique() for r in results.values()]
    rows.append({
        "Model": label,
        "Mean Scrutiny": f"{np.mean(s):.3f}",
        "Mean Value": f"${np.mean(v):,.0f}",
        "Topic Diversity": f"{np.mean(d):.1f}",
    })

rows.append({
    "Model": "Dataset Baseline",
    "Mean Scrutiny": f"{contracts['doge_scrutiny_score'].mean():.3f}",
    "Mean Value": f"${contracts['value'].mean():,.0f}",
    "Topic Diversity": "-",
})

pd.DataFrame(rows).set_index("Model")

## 5. Model Overlap (Jaccard)

In [ ]:
overlap_rows = []
for p in PERSONAS:
    c_ids = set(classical_results[p["name"]]["contract_id"])
    d_ids = set(dl_results[p["name"]]["contract_id"])
    jaccard = len(c_ids & d_ids) / len(c_ids | d_ids) if (c_ids | d_ids) else 0
    overlap_rows.append({
        "Persona": p["name"],
        "Topics": ", ".join(p["topics"]),
        "Jaccard": f"{jaccard:.3f}",
        "Shared": len(c_ids & d_ids),
    })

pd.DataFrame(overlap_rows).set_index("Persona")

## 6. Sample Recommendations (Healthcare Advocate)

In [ ]:
SHOW_COLS = ["contract_id", "agency", "topic", "doge_scrutiny_score", "value", "description"]
persona = "Healthcare advocate"

print("=== NAIVE BASELINE (same for all users) ===")
display(naive_top20[SHOW_COLS].head(5))

print("\n=== CLASSICAL (TF-IDF) ===")
display(classical_results[persona][SHOW_COLS].head(5))

print("\n=== DEEP LEARNING (MLP) ===")
display(dl_results[persona][SHOW_COLS].head(5))